<div style=" background-color: RGB(0,114,200);" >
<h2 style="margin: auto; padding: 20px 0; color:#fff; text-align: center">Détection de faux billets - Version fonctionnelle</h2>
</div>

Ce notebook contient une fonction complète `detecter_faux_billets()` qui permet de traiter facilement n'importe quel fichier de données avec n'importe quel modèle entraîné en amont en format pickle.

In [1]:
# Importation des librairies nécessaires
import pickle
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Définition de la fonction pour détecter les faux billets

In [2]:
def detecter_faux_billets(nom_modele_pkl, nom_fichier_csv, chemin_base=None):
    """
    Fonction complète pour détecter les faux billets
    
    Paramètres:
    - nom_modele_pkl: nom du fichier modèle (ex: "modele_regression_logistique_faux_billets.pkl")
    - nom_fichier_csv: nom du fichier de données (ex: "billets_production.csv")
    - chemin_base: chemin de base (optionnel)
    
    Retourne:
    - DataFrame avec les résultats et les prédictions
    """
    
    # Définition du chemin de base 
    if chemin_base is None:
        chemin_base = r"C:\Users\CRFP4534\OneDrive - orange.com\Bureau\Formation CFA - Data Analyst\P10-Détectez des faux billets avec R ou Python\\"
    
    print("🔍 Détection de faux billets - Début du processus")
    print("=" * 50)
    
    try:
        # 1. Chargement des données
        print("📁 Chargement des données...")
        chemin_csv = chemin_base + f"Data\\{nom_fichier_csv}"
        df = pd.read_csv(chemin_csv, delimiter=",")
        print(f"   ✅ Fichier chargé : {nom_fichier_csv}")
        
        # 2. Chargement du modèle
        print("🤖 Chargement du modèle...")
        chemin_modele = chemin_base + f"Livrables\\{nom_modele_pkl}"
        with open(chemin_modele, 'rb') as fichier:
            modele_charge = pickle.load(fichier)
        print(f"   ✅ Modèle chargé : {nom_modele_pkl}")
        
        # 3. Vérification de la structure des données
        print("\n📊 Structure des données de production :")
        print(f"   • Nombre de lignes : {df.shape[0]}")
        print(f"   • Nombre de colonnes : {df.shape[1]}")
        print(f"   • Colonnes disponibles : {list(df.columns)}")
        
        # 4. Préparation des données
        print("\n🔧 Préparation des données...")
        colonnes_requises = ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length']
        
        # Vérification que toutes les colonnes requises sont présentes
        colonnes_manquantes = [col for col in colonnes_requises if col not in df.columns]
        if colonnes_manquantes:
            raise ValueError(f"❌ Colonnes manquantes : {colonnes_manquantes}")
        else:
            print("   ✅ Toutes les colonnes requises sont présentes")
        
        # Réorganisation des données dans l'ordre requis
        df_model = df[colonnes_requises].copy()
        
        # Vérification des valeurs manquantes
        valeurs_manquantes = df_model.isnull().sum()
        if valeurs_manquantes.any():
            print(f"\n   ⚠️ Valeurs manquantes détectées :")
            for col, nb in valeurs_manquantes[valeurs_manquantes > 0].items():
                print(f"      • {col} : {nb} valeur(s) manquante(s)")
            print("   ⚠️ Les valeurs manquantes doivent être traitées pour avoir des résultats fiables.")
        else:
            print("   ✅ Aucune valeur manquante")
        
        # 5. Standardisation des données
        print("\n🔧 Standardisation des données...")
        scaler = StandardScaler()
        df_model_scaled = scaler.fit_transform(df_model)
        print("   ✅ Données standardisées avec succès")
        
        # 6. Prédictions
        print("\n🎯 Génération des prédictions...")
        predictions = modele_charge.predict(df_model_scaled)
        probabilites = modele_charge.predict_proba(df_model_scaled)
        
        # 7. Statistiques globales
        nb_vrais = sum(predictions)
        nb_faux = len(predictions) - nb_vrais
        pourcentage_vrais = (nb_vrais / len(predictions)) * 100
        
        print("\n📈 Statistiques globales :")
        print(f"   • Total analysé : {len(predictions)} billet(s)")
        print(f"   • Vrais billets : {nb_vrais} ({pourcentage_vrais:.1f}%)")
        print(f"   • Faux billets  : {nb_faux} ({100-pourcentage_vrais:.1f}%)")
        
        # 8. Préparation des résultats
        print("\n📋 Préparation des résultats...")
        resultats_csv = df.copy()
        
        # Ajout des colonnes de prédiction
        resultats_csv['probabilite_vrai'] = [prob[1] for prob in probabilites]
        resultats_csv['prediction'] = ['Vrai' if pred == 1 else 'Faux' for pred in predictions]
        
        # 9. Export des résultats
        print("💾 Export des résultats...")
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        nom_fichier_resultat = f"resultats_faux_billets_{timestamp}.csv"
        chemin_resultat = chemin_base + f"Livrables\\{nom_fichier_resultat}"
        
        resultats_csv.to_csv(chemin_resultat, index=False, encoding='utf-8-sig')
        
        print(f"   ✅ Export réalisé avec succès : {nom_fichier_resultat}")
        print("\n" + "=" * 50)
        print("🎉 Processus terminé avec succès !")
        
        return resultats_csv
        
    except Exception as e:
        print(f"\n❌ Erreur lors du traitement : {str(e)}")
        return None

# Application de la fonction avec les paramètres requis

In [3]:
# Renseignement des paramètres requis
nom_modele = "modele_regression_logistique_faux_billets.pkl"
nom_fichier_donnees = "billets_test.csv"

# Appel de la fonction avec le scaler
resultats = detecter_faux_billets(nom_modele, nom_fichier_donnees)

# Affichage des premiers résultats
if resultats is not None:
    print("\n📋 Aperçu des résultats :")
    print(resultats.head())

🔍 Détection de faux billets - Début du processus
📁 Chargement des données...
   ✅ Fichier chargé : billets_test.csv
🤖 Chargement du modèle...
   ✅ Modèle chargé : modele_regression_logistique_faux_billets.pkl

📊 Structure des données de production :
   • Nombre de lignes : 5
   • Nombre de colonnes : 7
   • Colonnes disponibles : ['diagonal', 'height_left', 'height_right', 'margin_low', 'margin_up', 'length', 'id']

🔧 Préparation des données...
   ✅ Toutes les colonnes requises sont présentes
   ✅ Aucune valeur manquante

🔧 Standardisation des données...
   ✅ Données standardisées avec succès

🎯 Génération des prédictions...

📈 Statistiques globales :
   • Total analysé : 5 billet(s)
   • Vrais billets : 2 (40.0%)
   • Faux billets  : 3 (60.0%)

📋 Préparation des résultats...
💾 Export des résultats...
   ✅ Export réalisé avec succès : resultats_faux_billets_20250801_142105.csv

🎉 Processus terminé avec succès !

📋 Aperçu des résultats :
   diagonal  height_left  height_right  margin_lo